# Pruebas de justificación con Qwen2.5 7B

Se prueban los casos `C001` y `C002` en español e inglés con Qwen2.5 7B usando la segunda versión de las instrucciones.

Ejecuta las celdas en orden con Ollama abierto en la carpeta donde están `casos.json` e `inventario_22_llm.csv`. Los resultados se guardan en `resultados.csv`.

## 1. Cargar los casos y el modelo

In [1]:
import os
import csv
import json
import time
from copy import deepcopy

import requests

id_modelo = "M01"
archivo_salida = "resultados.csv"
columnas = [
    "id_caso", "id_modelo", "idioma", "intento", "solicitud_json", "estado",
    "tiempo_s", "respuesta", "error", "evaluacion", "motivo_evaluacion"
]

# Cargar casos
with open("casos.json", "r", encoding="utf-8-sig") as f:
    datos = json.load(f)

casos = {}
for c in datos["casos"]:
    casos[c["id_caso"]] = c

# Buscar el modelo en el inventario
nombre_modelo = ""
etiqueta = ""
with open("inventario_22_llm.csv", "r", encoding="utf-8-sig", newline="") as f:
    for fila in csv.DictReader(f):
        if fila["id_modelo"] == id_modelo:
            nombre_modelo = fila["nombre_modelo"]
            etiqueta = fila["etiqueta_ollama"]
            break

# Crear el CSV de resultados con encabezados si aún no existe
if not os.path.exists(archivo_salida):
    with open(archivo_salida, "w", encoding="utf-8-sig", newline="") as f:
        csv.DictWriter(f, fieldnames=columnas).writeheader()

print("Modelo:", nombre_modelo, "|", etiqueta)

Modelo: Qwen2.5 7B | qwen2.5:7b


## 2. Preparar la solicitud

In [2]:
instrucciones = """Genera una justificación semántica de la predicción de coautoría recibida.

1. Indica el resultado de la predicción de coautoría.
2. Identifica las coincidencias y diferencias entre las publicaciones de ambos autores.
3. Explica cómo se utiliza cada concepto o método compartido en los artículos.
4. Cita los identificadores de los artículos que respaldan cada afirmación.
5. Indica si la información disponible permite justificar semánticamente la predicción.

Consideraciones:
- No modifiques la predicción recibida.
- No atribuyas las coincidencias encontradas a las causas de la predicción.
- Distingue la similitud entre artículos de la probabilidad de coautoría.
- No califiques los valores de similitud sin un criterio establecido.
- La ausencia de palabras clave no significa que no existan coincidencias.
- Si faltan probabilidades o evidencia, indícalo.
- Utiliza únicamente la información proporcionada."""

def preparar_solicitud(id_caso, idioma):
    entrada = deepcopy(casos[id_caso]["entrada_llm"])
    umbral_general = datos["regla_coautoria"]["umbral_definitivo"]

    for metodo in entrada["prediccion_coautoria"]["resultados_por_metodo"]:
        p = metodo.get("probabilidad_coautoria")
        umbral = metodo.get("umbral_aplicado")
        if umbral is None:
            umbral = umbral_general

        metodo["umbral_aplicado"] = umbral
        if p is not None and umbral is not None:
            metodo["clasificacion_por_umbral"] = "favorable" if p >= umbral else "desfavorable"
        else:
            metodo["clasificacion_por_umbral"] = None

    idioma_salida = datos["idiomas"]["instruccion_por_idioma"][idioma]

    return {
        "model": etiqueta,
        "messages": [
            {"role": "system", "content": instrucciones + "\n" + idioma_salida},
            {"role": "user", "content": json.dumps(entrada, ensure_ascii=False, indent=2)}
        ],
        "stream": False,
        "options": {"temperature": 0, "num_ctx": 4096, "num_predict": 1024}
    }

## 3. Pedir la justificación a Ollama

In [3]:
url = "http://localhost:11434/api"

def pedir_justificacion(solicitud):
    inicio = time.time()
    registro = {"estado": "error", "respuesta": "", "error": ""}

    try:
        r = requests.post(url + "/chat", json=solicitud, timeout=900)
        r.raise_for_status()
        salida = r.json()

        texto = salida["message"]["content"]
        registro["respuesta"] = texto

        if salida.get("done_reason") == "length":
            registro["estado"] = "truncada"
        elif salida.get("done") and texto.strip() != "":
            registro["estado"] = "completa"
        else:
            registro["estado"] = "incompleta"

    except Exception as e:
        registro["error"] = str(e)

    registro["tiempo_s"] = round(time.time() - inicio, 2)
    return registro

## 4. Ejecutar C001 y C002 en español e inglés

In [4]:
# Verificar que el modelo esté instalado en Ollama
r_tags = requests.get(url + "/tags", timeout=10)
r_tags.raise_for_status()
instalados = [m["name"].lower() for m in r_tags.json()["models"]]

if etiqueta.lower() not in instalados:
    print("Falta instalar el modelo en Ollama. Ejecuta: ollama pull", etiqueta)
else:
    for id_caso in ["C001", "C002"]:
        intento = 0
        for idioma in ["es", "en"]:
            intento += 1
            solicitud = preparar_solicitud(id_caso, idioma)

            print(f"\n=== {id_caso} | {idioma} | intento {intento} ===")
            resultado = pedir_justificacion(solicitud)

            fila = {
                "id_caso": id_caso,
                "id_modelo": id_modelo,
                "idioma": idioma,
                "intento": intento,
                "solicitud_json": json.dumps(solicitud, ensure_ascii=False),
                "estado": resultado["estado"],
                "tiempo_s": resultado["tiempo_s"],
                "respuesta": resultado["respuesta"],
                "error": resultado["error"],
                "evaluacion": "",
                "motivo_evaluacion": ""
            }

            with open(archivo_salida, "a", encoding="utf-8", newline="") as f:
                csv.DictWriter(f, fieldnames=columnas).writerow(fila)

            print(f"{fila['estado']} | {fila['tiempo_s']} s", fila["error"])


=== C001 | es | intento 1 ===
completa | 19.61 s 

=== C001 | en | intento 2 ===
completa | 11.77 s 

=== C002 | es | intento 1 ===
completa | 11.13 s 

=== C002 | en | intento 2 ===
completa | 9.96 s 
